In [1]:
import numpy as np
from PIL import Image
import skimage as ski
from scipy.ndimage import correlate, convolve, minimum_filter, maximum_filter
from skimage.restoration import denoise_bilateral
from skimage import img_as_float
from skimage.filters import gaussian, sobel, prewitt, laplace
from skimage.filters.rank import mean, median
import matplotlib.pyplot as plt

In [2]:
def low_pass_fourier_filter(img, cutoff=50):

    row, col = img.shape
    center_row, center_col = row // 2, col // 2

    fft_img = np.fft.fft2(img)
    fft_shift = np.fft.fftshift(fft_img)

    # Creating a fourier mask
    x = np.arange(col) - center_col
    y = np.arange(row) - center_row
    X, Y = np.meshgrid(x, y)

    # We preserve the central area, rejecting the outer part
    mask = np.sqrt(X**2 + Y**2) <= cutoff

    fft_filt = fft_shift * mask

    # 5. Deshacer el desplazamiento central
    fft_ishift = np.fft.ifftshift(fft_filt)
    img_filt_compl = np.fft.ifft2(fft_ishift)

    # The image must be real, so we take the real part
    img_filt = np.real(img_filt_compl)

    return img_filt

def high_pass_fourier_filter(img, cutoff=50):

    row, col = img.shape
    center_row, center_col = row // 2, col // 2

    fft_img = np.fft.fft2(img)
    fft_shift = np.fft.fftshift(fft_img)

    # Creating a fourier mask
    x = np.arange(col) - center_col
    y = np.arange(row) - center_row
    X, Y = np.meshgrid(x, y)

    # We preserve the central area, rejecting the outer part
    mask = np.sqrt(X**2 + Y**2) >= cutoff

    fft_filt = fft_shift * mask

    fft_ishift = np.fft.ifftshift(fft_filt)
    img_filt_compl = np.fft.ifft2(fft_ishift)

    # The image must be real, so we take the real part
    img_filt = np.real(img_filt_compl)

    return img_filt

def band_pass_fourier_filter(img, low_cutoff=30, high_cutoff=80):

    row, col = img.shape
    center_row, center_col = row // 2, col // 2

    fft_img = np.fft.fft2(img)
    fft_shift = np.fft.fftshift(fft_img)

    # Creating a fourier mask
    x = np.arange(col) - center_col
    y = np.arange(row) - center_row
    X, Y = np.meshgrid(x, y)

    # Calculate distance from the center for every pixel
    distance = np.sqrt(X**2 + Y**2)

    # We preserve the area BETWEEN low_cutoff and high_cutoff (creating a ring mask)
    mask = (distance >= low_cutoff) & (distance <= high_cutoff)

    fft_filt = fft_shift * mask

    # Undo the central shift
    fft_ishift = np.fft.ifftshift(fft_filt)
    img_filt_compl = np.fft.ifft2(fft_ishift)

    # The image must be real, so we take the real part
    img_filt = np.real(img_filt_compl)

    return img_filt


def directional_fourier_filter(img, angle=0, width=10):

    row, col = img.shape
    center_row, center_col = row // 2, col // 2

    # Compute the 2D Fast Fourier Transform
    fft_img = np.fft.fft2(img)
    fft_shift = np.fft.fftshift(fft_img)

    x = np.arange(col) - center_col
    y = center_row - np.arange(row)
    X, Y = np.meshgrid(x, y)

    # Calculate the angle of each coordinate in degrees [-180, 180]
    theta = np.degrees(np.arctan2(Y, X))

    # Calculate the shortest angular distance from our target angle.
    # We use modulo 180 because the Fourier spectrum of a real image is symmetric.
    # A line at 0 degrees is the same as a line at 180 degrees.
    angular_diff = np.abs((theta - angle + 90) % 180 - 90)

    mask = angular_diff <= (width / 2.0)
    mask[center_row, center_col] = True
    fft_filt = fft_shift * mask

    fft_ishift = np.fft.ifftshift(fft_filt)
    img_filt_compl = np.fft.ifft2(fft_ishift)

    img_filt = np.real(img_filt_compl)

    return img_filt

In [3]:
# read image
fnImg = 'sample_data/parlamentoHungria.jpeg'
img = Image.open(fnImg)

# Convert to numpy array
img_array = np.array(img)
gray_img = np.array(img.convert('L'))

img_filt_low  = low_pass_fourier_filter(gray_img, cutoff=50)
img_filt_high = high_pass_fourier_filter(gray_img, cutoff=50)
img_filt_band = band_pass_fourier_filter(gray_img, low_cutoff=30, high_cutoff=80)
img_filt_dir  = directional_fourier_filter(gray_img, angle=0, width=30)

plt.figure(figsize=(12, 4))
plt.subplot(1, 5, 1)
plt.title("Original Image")
plt.imshow(gray_img, cmap='gray')
plt.axis('off')

plt.subplot(1, 5, 2)
plt.title("low pass filtered image")
plt.imshow(img_filt_low, cmap='gray')
plt.axis('off')

plt.subplot(1, 5, 3)
plt.title("high pass filtered image")
plt.imshow(img_filt_high, cmap='gray')
plt.axis('off')

plt.subplot(1, 5, 4)
plt.title("band pass filtered image")
plt.imshow(img_filt_band, cmap='gray')
plt.axis('off')

plt.subplot(1, 5, 5)
plt.title("directional filtered image")
plt.imshow(img_filt_dir, cmap='gray')
plt.axis('off')



FileNotFoundError: [Errno 2] No such file or directory: 'sample_data/parlamentoHungria.jpeg'